# Biomedical Data Warehouse & Analytical SQL -- TCGA-BRCA + Open Targets

Build a real, normalized DuckDB warehouse from two independent real sources -- real TCGA breast cancer
clinical/mutation data (cBioPortal) and real drug-target association data (Open Targets) -- then answer real
biological questions purely in SQL: which genes are most frequently mutated in this real cohort, which of
those have real known drugs already targeting them, and whether a gene's real mutation status associates
with real patient survival.

~80-90% of the code below is left blank (`____`) for you to write from each cell's hint + the library/API
docs. Read `PROJECT_NARRATIVE.md` and `WORKFLOW.md` first (private, not in this repo's public version).

CPU-only, no GPU, no model training -- this is a data-engineering project. Should run in well under an hour.

## Step 0: Environment setup (GIVEN)
True boilerplate only -- installs + imports.

In [36]:
!pip install -q duckdb requests pandas

import os
import requests
import pandas as pd
import duckdb

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

CBIOPORTAL_BASE = "https://www.cbioportal.org/api"
STUDY_ID = "brca_tcga_pan_can_atlas_2018"  # real TCGA Breast Invasive Carcinoma, PanCancer Atlas 2018

print("duckdb version:", duckdb.__version__)

duckdb version: 1.5.6


## Step 1: Fetch real BRCA clinical data and pivot to a wide patients table
**WHAT:** fetch real per-patient clinical attributes (survival time/status, tumor stage, ethnicity, ...) for
the real TCGA-BRCA PanCancer Atlas cohort.
**WHY:** this becomes the real `patients` dimension table in the warehouse.
**HOW:** `GET /studies/{studyId}/clinical-data?clinicalDataType=PATIENT&projection=SUMMARY` -- confirmed live
to return a real LONG-format list of `{patientId, clinicalAttributeId, value}` records, NOT already a wide
table -- you need to pivot it yourself (`pandas.pivot_table` or `.pivot`) into one real row per patient.

In [37]:
# WRITE-IT: fetch the real clinical data for this study (paginate if needed -- check the real response
# length against the study's real patient count; cBioPortal may cap a single page, use pageSize/pageNumber
# params if so)
# hint:
#   resp = requests.get(f"{CBIOPORTAL_BASE}/studies/{STUDY_ID}/clinical-data",
#                        params={"clinicalDataType": "PATIENT", "projection": "SUMMARY", "pageSize": 10000})
#   resp.raise_for_status()
#   clinical_long = pd.DataFrame(resp.json())
clinical_records = []
page_number = 0
page_size = 10000
while True:
    resp = requests.get(
        f"{CBIOPORTAL_BASE}/studies/{STUDY_ID}/clinical-data",
        params={"clinicalDataType": "PATIENT", "projection": "SUMMARY",
                "pageSize": page_size, "pageNumber": page_number}
    )
    resp.raise_for_status()
    page_data = resp.json()
    clinical_records.extend(page_data)
    if len(page_data) < page_size:
        break
    page_number += 1

clinical_long = pd.DataFrame(clinical_records)
print(clinical_long.shape)
print(clinical_long["patientId"].nunique())

print(clinical_long.shape)
print(clinical_long.head())
print("real unique clinicalAttributeId values:", clinical_long["clinicalAttributeId"].unique()[:20])

# WRITE-IT: pivot the real long-format data to one row per real patient (columns = clinicalAttributeId,
# values = value), keyed on patientId
# hint: patients_wide = clinical_long.pivot_table(index="patientId", columns="clinicalAttributeId",
#           values="value", aggfunc="first").reset_index()
patients_wide = clinical_long.pivot_table(index="patientId", columns="clinicalAttributeId", values="value", aggfunc="first").reset_index()
print(patients_wide.shape)
print(patients_wide[["patientId", "DSS_MONTHS", "DSS_STATUS"]].head())

(41497, 5)
1084
(41497, 5)
                                    uniquePatientKey     patientId  \
0  VENHQS0zQy1BQUFVOmJyY2FfdGNnYV9wYW5fY2FuX2F0bG...  TCGA-3C-AAAU   
1  VENHQS0zQy1BQUFVOmJyY2FfdGNnYV9wYW5fY2FuX2F0bG...  TCGA-3C-AAAU   
2  VENHQS0zQy1BQUFVOmJyY2FfdGNnYV9wYW5fY2FuX2F0bG...  TCGA-3C-AAAU   
3  VENHQS0zQy1BQUFVOmJyY2FfdGNnYV9wYW5fY2FuX2F0bG...  TCGA-3C-AAAU   
4  VENHQS0zQy1BQUFVOmJyY2FfdGNnYV9wYW5fY2FuX2F0bG...  TCGA-3C-AAAU   

                        studyId          clinicalAttributeId    value  
0  brca_tcga_pan_can_atlas_2018                          AGE       55  
1  brca_tcga_pan_can_atlas_2018  AJCC_PATHOLOGIC_TUMOR_STAGE  STAGE X  
2  brca_tcga_pan_can_atlas_2018         AJCC_STAGING_EDITION      6TH  
3  brca_tcga_pan_can_atlas_2018          BUFFA_HYPOXIA_SCORE      -21  
4  brca_tcga_pan_can_atlas_2018          CANCER_TYPE_ACRONYM     BRCA  
real unique clinicalAttributeId values: ['AGE' 'AJCC_PATHOLOGIC_TUMOR_STAGE' 'AJCC_STAGING_EDITION'
 'BUFFA_HYPOXIA_SCOR

## Step 2: Fetch real BRCA mutation data -- verify the real live response shape first
**WHAT:** fetch real somatic mutation records (gene, sample, mutation type) for this real cohort.
**WHY:** this becomes the real `mutations` fact table.
**HOW / IMPORTANT:** the real mutation-fetch endpoint is
`POST /molecular-profiles/{molecularProfileId}/mutations/fetch` with a real JSON body
`{"sampleListId": "..."}` -- this specific endpoint's exact live response shape was NOT independently
verified in advance (a sandbox limitation, disclosed in WORKFLOW.md) -- print the raw real response first and
inspect its real structure before writing any parsing logic. If this exact shape doesn't match, check
`cbioportal.org/api/swagger-ui/index.html` directly for the real current request/response schema.

In [38]:
MUTATION_PROFILE_ID = f"{STUDY_ID}_mutations"           # confirmed real: brca_tcga_pan_can_atlas_2018_mutations
SAMPLE_LIST_ID = f"{STUDY_ID}_sequenced"                  # confirmed real: brca_tcga_pan_can_atlas_2018_sequenced

# WRITE-IT: POST to the real mutations/fetch endpoint and print the RAW response first to verify its real
# shape before assuming any field names
# hint:
#   resp = requests.post(
#       f"{CBIOPORTAL_BASE}/molecular-profiles/{MUTATION_PROFILE_ID}/mutations/fetch",
#       params={"projection": "SUMMARY"},
#       json={"sampleListId": SAMPLE_LIST_ID})
#   resp.raise_for_status()
#   print(resp.status_code)
#   print(resp.json()[:3])   # print a few RAW real records before trusting any column name
resp = requests.post(f"{CBIOPORTAL_BASE}/molecular-profiles/{MUTATION_PROFILE_ID}/mutations/fetch", params={"projection":"DETAILED"}, json={"sampleListId":SAMPLE_LIST_ID})
resp.raise_for_status()
print(resp.status_code)
print(resp.json() if resp.status_code != 200 else resp.json()[:3])

# WRITE-IT: once you've confirmed the real shape above, build a real mutations DataFrame
# (expect real fields along the lines of: sampleId, patientId, hugoGeneSymbol (nested under "gene" or
# top-level -- check what you actually printed above), proteinChange, mutationType)
mutations_df = pd.DataFrame(resp.json())
mutations_df["hugoGeneSymbol"] = mutations_df["gene"].apply(lambda g: g["hugoGeneSymbol"])

200
[{'uniqueSampleKey': 'VENHQS0zQy1BQUFVLTAxOmJyY2FfdGNnYV9wYW5fY2FuX2F0bGFzXzIwMTg', 'uniquePatientKey': 'VENHQS0zQy1BQUFVOmJyY2FfdGNnYV9wYW5fY2FuX2F0bGFzXzIwMTg', 'molecularProfileId': 'brca_tcga_pan_can_atlas_2018_mutations', 'sampleId': 'TCGA-3C-AAAU-01', 'patientId': 'TCGA-3C-AAAU', 'entrezGeneId': 908, 'gene': {'entrezGeneId': 908, 'hugoGeneSymbol': 'CCT6A', 'type': 'protein-coding'}, 'studyId': 'brca_tcga_pan_can_atlas_2018', 'center': '.', 'mutationStatus': '.', 'validationStatus': '.', 'tumorAltCount': 22, 'tumorRefCount': 85, 'normalAltCount': 0, 'normalRefCount': 122, 'chr': '7', 'startPosition': 56130450, 'endPosition': 56130450, 'referenceAllele': 'T', 'variantAllele': 'C', 'proteinChange': 'X508_splice', 'mutationType': 'Splice_Site', 'ncbiBuild': 'GRCh37', 'variantType': 'SNP', 'refseqMrnaId': 'NM_001762.3', 'proteinPosStart': 508, 'proteinPosEnd': 508, 'keyword': 'CCT6A truncating'}, {'uniqueSampleKey': 'VENHQS0zQy1BQUFVLTAxOmJyY2FfdGNnYV9wYW5fY2FuX2F0bGFzXzIwMTg', 'u

In [39]:
print(mutations_df.shape)
print(mutations_df[["sampleId", "patientId", "hugoGeneSymbol", "mutationType", "proteinChange"]].head())

(84226, 29)
          sampleId     patientId hugoGeneSymbol       mutationType  \
0  TCGA-3C-AAAU-01  TCGA-3C-AAAU          CCT6A        Splice_Site   
1  TCGA-3C-AAAU-01  TCGA-3C-AAAU          CDC20  Missense_Mutation   
2  TCGA-3C-AAAU-01  TCGA-3C-AAAU          EP300    Frame_Shift_Ins   
3  TCGA-3C-AAAU-01  TCGA-3C-AAAU          GATA3    Frame_Shift_Ins   
4  TCGA-3C-AAAU-01  TCGA-3C-AAAU         GOLGA2  Missense_Mutation   

  proteinChange  
0   X508_splice  
1         H410Y  
2    M1339Ifs*2  
3    P408Afs*99  
4         A167V  


## Step 3: Design and build the real normalized DuckDB schema
**WHAT:** create real `patients`, `samples`, and `mutations` tables with real primary/foreign keys, and load
the real fetched data into them.
**WHY:** this is the actual data-engineering deliverable -- a real, queryable relational warehouse, not just
two DataFrames sitting in memory.
**HOW:** `duckdb.connect("warehouse.duckdb")`, then `CREATE TABLE ... (col TYPE, ..., PRIMARY KEY(...))` SQL
run via `con.execute(...)`, then `con.register("df_name", df)` + `INSERT INTO ... SELECT * FROM df_name`
(or `con.sql("CREATE TABLE t AS SELECT * FROM df")` for a quick direct load, if you don't need strict
typed DDL for every column).

In [40]:
con = duckdb.connect("warehouse.duckdb")

# WRITE-IT: create a real patients table from patients_wide (register the DataFrame, then CREATE TABLE ... AS
# SELECT the real columns you actually need: patientId, DSS_MONTHS, DSS_STATUS, and a couple more real
# clinical attributes of your choice from Step 1's printed unique list)
# hint:
#   con.register("patients_df", patients_wide)
#   con.execute("CREATE OR REPLACE TABLE patients AS SELECT patientId AS patient_id, DSS_MONTHS, DSS_STATUS "
#               "FROM patients_df")
con.register("patients_df", patients_wide)
con.execute("CREATE OR REPLACE TABLE patients AS SELECT patientId AS patient_id, DSS_MONTHS, DSS_STATUS FROM patients_df")
# WRITE-IT: create a real mutations table from mutations_df (keep patientId/sampleId, the real gene symbol
# column, and the real mutation-type/protein-change columns you found in Step 2)
# hint: con.register("mutations_df_raw", mutations_df)
#       con.execute("CREATE OR REPLACE TABLE mutations AS SELECT * FROM mutations_df_raw")
con.register("mutations_df_raw", mutations_df)
con.execute("CREATE OR REPLACE TABLE mutations AS SELECT * FROM mutations_df_raw")

print(con.execute("SELECT COUNT(*) FROM patients").fetchall())
print(con.execute("SELECT COUNT(*) FROM mutations").fetchall())

[(1084,)]
[(84226,)]


## Step 4: Real analytical SQL -- rank genes by real mutation frequency (window function)
**WHAT:** which real genes are mutated in the most real samples in this cohort, with an explicit real rank
(not just a sorted count) using a window function.
**HOW:** `RANK() OVER (ORDER BY mutation_count DESC)` on top of a `GROUP BY` subquery/CTE.

In [41]:
# WRITE-IT: write and run a real SQL query that counts DISTINCT samples per gene, then ranks genes by that
# count using RANK() OVER (...) -- use the real gene-symbol column name you found in Step 2
# hint:
#   gene_rank_sql = """
#       WITH gene_counts AS (
#           SELECT hugoGeneSymbol AS gene_symbol, COUNT(DISTINCT sampleId) AS mutated_sample_count
#           FROM mutations
#           GROUP BY hugoGeneSymbol
#       )
#       SELECT gene_symbol, mutated_sample_count,
#              RANK() OVER (ORDER BY mutated_sample_count DESC) AS mutation_rank
#       FROM gene_counts
#       ORDER BY mutation_rank
#   """
#   gene_ranking = con.execute(gene_rank_sql).fetchdf()
gene_rank_sql = """ 
    WITH gene_counts AS (
        SELECT hugoGeneSymbol AS gene_Symbol, COUNT(DISTINCT sampleId) AS mutated_sample_count
        FROM mutations
        GROUP BY hugoGeneSymbol
    )
    SELECT gene_Symbol, mutated_sample_count,
           RANK() OVER (ORDER BY mutated_sample_count DESC) AS mutation_rank
    FROM gene_counts
    ORDER BY mutation_rank
"""
gene_ranking = con.execute(gene_rank_sql).fetchdf()
print(gene_ranking.head(15))

# WRITE-IT: pull out the real top 10 gene symbols as a plain Python list for Step 5
# hint: top_genes = gene_ranking.head(10)["gene_symbol"].tolist()
top_genes = gene_ranking.head(10)["gene_Symbol"].tolist()
print("real top mutated genes:", top_genes)

   gene_Symbol  mutated_sample_count  mutation_rank
0         TP53                   347              1
1       PIK3CA                   347              1
2          TTN                   187              3
3         CDH1                   130              4
4        GATA3                   127              5
5        MUC16                   109              6
6        KMT2C                    99              7
7       MAP3K1                    89              8
8         RYR2                    66              9
9          FLG                    66              9
10       SYNE1                    63             11
11       USH2A                    62             12
12       HMCN1                    61             13
13        PTEN                    58             14
14       SPTA1                    54             15
real top mutated genes: ['TP53', 'PIK3CA', 'TTN', 'CDH1', 'GATA3', 'MUC16', 'KMT2C', 'MAP3K1', 'RYR2', 'FLG']


## Step 5: Fetch real known-drug data from Open Targets for the real top mutated genes
**WHAT:** for each real top-mutated gene, query Open Targets for real drugs with known activity against it.
**HOW:** Open Targets' public GraphQL API (`https://api.platform.opentargets.org/api/v4/graphql`) -- first
resolve each real gene symbol to its real Ensembl target ID via a `search` query, then query that target's
real `knownDrugs` field (same real API already used successfully in this portfolio's #24 project).

In [42]:
OPEN_TARGETS_URL = "https://api.platform.opentargets.org/api/v4/graphql"

SEARCH_QUERY = """
query search($q: String!) {
  search(queryString: $q, entityNames: ["target"]) {
    hits { id name }
  }
}
"""

# Open Targets renamed Target.knownDrugs -> Target.drugAndClinicalCandidates in a schema update
# (confirmed live via introspection, not from docs, since third-party docs disagreed with each other).
# Real confirmed field chain:
#   Target.drugAndClinicalCandidates -> clinicalTargets { rows: [ClinicalTargetFromTarget] }
#   ClinicalTargetFromTarget: maxClinicalStage (String), drug (Drug)
#   Drug: name (String), mechanismsOfAction (MechanismsOfAction)
#   MechanismsOfAction: rows: [MechanismOfActionRow]
#   MechanismOfActionRow: mechanismOfAction (String)
KNOWN_DRUGS_QUERY = """
query knownDrugs($ensemblId: String!) {
  target(ensemblId: $ensemblId) {
    approvedSymbol
    drugAndClinicalCandidates {
      rows {
        maxClinicalStage
        drug {
          name
          mechanismsOfAction {
            rows {
              mechanismOfAction
            }
          }
        }
      }
    }
  }
}
"""

drug_rows = []
for gene_symbol in top_genes:
    search_resp = requests.post(OPEN_TARGETS_URL, json={"query": SEARCH_QUERY, "variables": {"q": gene_symbol}})
    hits = search_resp.json()["data"]["search"]["hits"]
    if not hits:
        continue
    ensembl_id = hits[0]["id"]

    drugs_resp = requests.post(OPEN_TARGETS_URL, json={"query": KNOWN_DRUGS_QUERY, "variables": {"ensemblId": ensembl_id}})
    drugs_json = drugs_resp.json()
    if "errors" in drugs_json:
        print(f"skipping {gene_symbol}: {drugs_json['errors']}")
        continue

    target_data = drugs_json["data"]["target"]
    if target_data is None:
        continue

    rows = target_data["drugAndClinicalCandidates"]["rows"] or []
    for row in rows:
        moa_rows = (row["drug"]["mechanismsOfAction"] or {}).get("rows", []) if row["drug"]["mechanismsOfAction"] else []
        mechanism_text = moa_rows[0]["mechanismOfAction"] if moa_rows else None
        drug_rows.append({
            "gene_symbol": gene_symbol,
            "drug_name": row["drug"]["name"],
            "mechanism_of_action": mechanism_text,
            "max_clinical_stage": row["maxClinicalStage"],
        })

gene_drug_df = pd.DataFrame(drug_rows).drop_duplicates()
print(gene_drug_df.shape)
print(gene_drug_df.head(15))

con.register("gene_drug_df_raw", gene_drug_df)
con.execute("CREATE OR REPLACE TABLE gene_drug_targets AS SELECT * FROM gene_drug_df_raw")

(46, 4)
   gene_symbol              drug_name  \
0         TP53             TEPRASIRAN   
1         TP53             SIREMADLIN   
2         TP53            IDASANUTLIN   
3         TP53            NAVTEMADLIN   
4         TP53               CENERSEN   
5         TP53           EPRENETAPOPT   
6         TP53  CONTUSUGENE LADENOVEC   
7         TP53        CENERSEN SODIUM   
8         TP53           ALRIZOMADLIN   
9       PIK3CA              PANULISIB   
10      PIK3CA            PILARALISIB   
11      PIK3CA              IZORLISIB   
12      PIK3CA                SF-1126   
13      PIK3CA             COPANLISIB   
14      PIK3CA                PKI-179   

                                 mechanism_of_action max_clinical_stage  
0                            p53 mRNA RNAi inhibitor            PHASE_3  
1   Tumour suppressor p53/oncoprotein Mdm2 inhibitor            PHASE_2  
2   Tumour suppressor p53/oncoprotein Mdm2 inhibitor            PHASE_3  
3   Tumour suppressor p53/oncoprotein M

## Step 6: Real analytical SQL -- join mutation frequency with real drug-target data
**WHAT:** the catalog's own headline query -- which frequently-mutated real genes already have real known
drugs targeting them.
**HOW:** a real SQL `JOIN` between the gene-ranking query (or a materialized version of it) and
`gene_drug_targets`.

In [43]:
# WRITE-IT: write and run a real SQL query joining ranked mutated genes with their real known drugs
# (reuse Step 4's gene_rank_sql as a CTE, or materialize gene_ranking as a real table first)
# hint:
con.register("gene_ranking_df", gene_ranking)
con.execute("CREATE OR REPLACE TABLE gene_ranking AS SELECT * FROM gene_ranking_df")

druggable_sql = """
    SELECT r.gene_symbol, r.mutated_sample_count, r.mutation_rank,
           d.drug_name, d.mechanism_of_action, d.max_clinical_stage
    FROM gene_ranking r
    JOIN gene_drug_targets d ON r.gene_symbol = d.gene_symbol
    ORDER BY r.mutation_rank, d.max_clinical_stage DESC
"""
druggable_genes = con.execute(druggable_sql).fetchdf()
print(druggable_genes.head(20))

   gene_Symbol  mutated_sample_count  mutation_rank              drug_name  \
0       PIK3CA                   347              1               AZD-6482   
1       PIK3CA                   347              1              ALPELISIB   
2       PIK3CA                   347              1            GEDATOLISIB   
3         TP53                   347              1            NAVTEMADLIN   
4       PIK3CA                   347              1              TASELISIB   
5         TP53                   347              1             TEPRASIRAN   
6       PIK3CA                   347              1             BUPARLISIB   
7         TP53                   347              1  CONTUSUGENE LADENOVEC   
8         TP53                   347              1            IDASANUTLIN   
9         TP53                   347              1           EPRENETAPOPT   
10      PIK3CA                   347              1             DACTOLISIB   
11      PIK3CA                   347              1             

## Step 7: Real analytical SQL -- does a top gene's mutation status associate with real survival?
**WHAT:** a genuine biological validation question -- for whichever gene ranked #1 by real mutation
frequency, does its real mutation status associate with real patient survival (DSS_MONTHS)?
**HOW:** a real `CASE WHEN EXISTS (...)` or `LEFT JOIN` to label each patient mutated/not-mutated for that
one real gene, then `GROUP BY` that label and `AVG(DSS_MONTHS)`.

In [44]:
TOP_GENE = top_genes[0]

# WRITE-IT: write and run a real SQL query comparing average real DSS_MONTHS between patients with vs.
# without a real mutation in TOP_GENE
# hint:
survival_sql = f"""
    SELECT
    CASE WHEN m.patient_id IS NOT NULL THEN 'mutated' ELSE 'not_mutated' END AS {TOP_GENE}_status,
    COUNT(DISTINCT p.patient_id) AS n_patients,
    AVG(TRY_CAST(p.DSS_MONTHS AS DOUBLE)) AS avg_dss_months
    FROM patients p
    LEFT JOIN (SELECT DISTINCT patientId AS patient_id FROM mutations WHERE hugoGeneSymbol = '{TOP_GENE}') m
        ON p.patient_id = m.patient_id
    GROUP BY 1
   """
survival_by_mutation = con.execute(survival_sql).fetchdf()
print(survival_by_mutation)

   TP53_status  n_patients  avg_dss_months
0  not_mutated         737       40.315801
1      mutated         347       41.961879


In [45]:
print(con.execute("SELECT COUNT(*) FROM patients").fetchall())
print(con.execute("SELECT COUNT(*) FROM mutations").fetchall())
print(clinical_long.shape)
print(clinical_long["patientId"].nunique())

[(1084,)]
[(84226,)]
(41497, 5)
1084


## Step 8: Export real results as CSV

In [46]:
# WRITE-IT: export the real gene ranking, the real druggable-gene join, and the real survival comparison
# hint: gene_ranking.to_csv(os.path.join(RESULTS_DIR, "gene_mutation_ranking.csv"), index=False)
#       druggable_genes.to_csv(os.path.join(RESULTS_DIR, "druggable_frequently_mutated_genes.csv"), index=False)
#       survival_by_mutation.to_csv(os.path.join(RESULTS_DIR, "survival_by_top_gene_mutation.csv"), index=False)
gene_ranking.to_csv(os.path.join(RESULTS_DIR, "gene_mutation_ranking.csv"), index=False)
druggable_genes.to_csv(os.path.join(RESULTS_DIR, "druggable_frequently_mutated_genes.csv"), index=False)
survival_by_mutation.to_csv(os.path.join(RESULTS_DIR, "survival_by_top_gene_mutation.csv"), index=False)

print("real results exported to", RESULTS_DIR)
con.close()

real results exported to results
